# Tennis Bird's-Eye Demo

Use the dropdown to pick a sample clip from `data/samples/`, then click **Render bird's-eye view**.

The notebook will:
- run or reuse the existing court + player + ball tracking
- extract player skeletons with a lightweight cached pose pass
- project detections into a normalized top-down tennis court
- display the real video and the bird's-eye video side by side
- show one preview frame pair below the videos
- print a small detection summary

All generated files go into `out/`.

Note: the bird's-eye ball is rendered as an event-aware court shadow. It uses detected serve / hit / bounce anchors plus monotonic shot progress, instead of raw per-frame ground-plane projection, which avoids the false forward-backward motion during airborne shots. The first render per clip is slower because pose extraction is cached to disk.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import Video, display

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tennis_tracker.birdseye import render_birdseye_video
from tennis_tracker.events import write_match_events_json
from tennis_tracker.pipeline import run_pipeline
from tennis_tracker.pose import extract_player_poses

SAMPLES_DIR = ROOT / "data" / "samples"
SAMPLE_VIDEOS = sorted(SAMPLES_DIR.glob("*.mp4"))
assert SAMPLE_VIDEOS, f"No sample videos found in {SAMPLES_DIR}"

DEFAULT_VIDEO = next((p for p in SAMPLE_VIDEOS if p.name == "broadcast_2s.mp4"), SAMPLE_VIDEOS[0])
WEIGHTS_PATH = ROOT / "models" / "tracknet_weights.pth"
POSE_MODEL = ROOT / "models" / "yolo11n-pose.pt"
if not POSE_MODEL.exists():
    POSE_MODEL = Path("yolo11n-pose.pt")

OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(exist_ok=True)

TRACKING_VIDEO = OUT_DIR / "all.mp4"
TRACKING_JSON = OUT_DIR / "all.json"
EVENTS_JSON = OUT_DIR / "all_events.json"
POSE_JSON = OUT_DIR / "birdseye_pose.json"
BIRDSEYE_VIDEO = OUT_DIR / "birdseye.mp4"
BIRDSEYE_META = OUT_DIR / "birdseye_meta.json"
BIRDSEYE_RENDER_VERSION = 2
SELECTED_VIDEO_PATH = OUT_DIR / "selected_video_birdseye.txt"

assert WEIGHTS_PATH.exists(), f"Missing TrackNet weights: {WEIGHTS_PATH}"

selector = widgets.Dropdown(
    options=[(video.name, str(video)) for video in SAMPLE_VIDEOS],
    value=str(DEFAULT_VIDEO),
    description="Video:",
    layout=widgets.Layout(width="430px"),
)
force_rerun = widgets.Checkbox(value=False, description="Force rerun")
render_button = widgets.Button(description="Render bird's-eye view", button_style="primary")
ui_output = widgets.Output()


def read_preview_frame(video_path: Path, frame_index: int) -> cv2.typing.MatLike:
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Could not read frame {frame_index} from {video_path}")
    return frame


def build_video_panel(video_path: Path, title: str) -> widgets.HTML:
    video = Video(
        filename=str(video_path),
        embed=True,
        width=560,
        html_attributes="controls autoplay loop muted playsinline",
    )
    video_html = video._repr_html_() or ""
    return widgets.HTML(
        value=(
            "<div style='width:100%'>"
            f"<div style='font-weight:600; margin-bottom:6px;'>{title}</div>"
            f"{video_html}"
            "</div>"
        ),
        layout=widgets.Layout(width="50%"),
    )


def render_birdseye(video_path: Path, *, force: bool = False) -> None:
    current_source = SELECTED_VIDEO_PATH.read_text().strip() if SELECTED_VIDEO_PATH.exists() else None
    render_meta = json.loads(BIRDSEYE_META.read_text()) if BIRDSEYE_META.exists() else {}
    needs_tracking = (
        force
        or not TRACKING_VIDEO.exists()
        or not TRACKING_JSON.exists()
        or current_source != video_path.name
    )
    needs_pose = (
        force
        or needs_tracking
        or not POSE_JSON.exists()
        or current_source != video_path.name
    )

    if needs_tracking:
        run_pipeline(
            video_path=video_path,
            output_dir=OUT_DIR,
            step="all",
            weights_path=WEIGHTS_PATH,
        )

    if needs_pose:
        extract_player_poses(
            video_path=video_path,
            tracking_json_path=TRACKING_JSON,
            output_json_path=POSE_JSON,
            model_name=POSE_MODEL,
            imgsz=256,
            batch_size=16,
        )
    if force or needs_tracking or not EVENTS_JSON.exists():
        write_match_events_json(
            tracking_json_path=TRACKING_JSON,
            output_json_path=EVENTS_JSON,
        )

    needs_birdseye = (
        force
        or needs_tracking
        or needs_pose
        or not EVENTS_JSON.exists()
        or not BIRDSEYE_VIDEO.exists()
        or render_meta.get("version") != BIRDSEYE_RENDER_VERSION
        or render_meta.get("source") != video_path.name
    )

    if needs_birdseye:
        render_birdseye_video(
            tracking_json_path=TRACKING_JSON,
            output_video_path=BIRDSEYE_VIDEO,
            pose_json_path=POSE_JSON,
            events_json_path=EVENTS_JSON,
            ball_strategy="event_aware",
        )
        BIRDSEYE_META.write_text(
            json.dumps(
                {
                    "source": video_path.name,
                    "version": BIRDSEYE_RENDER_VERSION,
                    "ball_strategy": "event_aware_monotonic",
                },
                indent=2,
            )
        )

    SELECTED_VIDEO_PATH.write_text(video_path.name)

    data = json.loads(TRACKING_JSON.read_text())
    pose_data = json.loads(POSE_JSON.read_text())
    cap = cv2.VideoCapture(str(BIRDSEYE_VIDEO))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    target_frame = min(10, max(0, frame_count - 1))

    source_frame = read_preview_frame(video_path, target_frame)
    birdseye_frame = read_preview_frame(BIRDSEYE_VIDEO, target_frame)

    pose_frame_count = sum(bool(frame_data.get("players")) for frame_data in pose_data["frames"])
    summary = {
        "selected_video": video_path.name,
        "frames": len(data["frames"]),
        "ball_detections": sum(frame_data["ball"]["image_xy"] is not None for frame_data in data["frames"]),
        "pose_frames": pose_frame_count,
        "players_in_first_frame": len(data["frames"][0]["players"]),
    }

    ui_output.clear_output(wait=True)
    with ui_output:
        print(summary)
        display(
            widgets.HBox(
                [
                    build_video_panel(video_path, "Source video"),
                    build_video_panel(BIRDSEYE_VIDEO, "Bird's-eye video"),
                ],
                layout=widgets.Layout(align_items="flex-start"),
            )
        )

        figure, axes = plt.subplots(1, 2, figsize=(13, 8))
        axes[0].imshow(cv2.cvtColor(source_frame, cv2.COLOR_BGR2RGB))
        axes[0].axis("off")
        axes[0].set_title(f"Source frame {target_frame}")
        axes[1].imshow(cv2.cvtColor(birdseye_frame, cv2.COLOR_BGR2RGB))
        axes[1].axis("off")
        axes[1].set_title(f"Bird's-eye frame {target_frame}")
        plt.tight_layout()
        plt.show()


In [2]:
def on_render_clicked(_: widgets.Button) -> None:
    render_birdseye(Path(selector.value), force=force_rerun.value)


render_button.on_click(on_render_clicked)
controls = widgets.HBox([selector, force_rerun, render_button])
display(widgets.VBox([controls, ui_output]))
render_birdseye(Path(selector.value), force=False)
